# Ensemble Methods


In [1]:
import pandas as pd

import numpy as np

## Choose your own Regression

**Working together with your partner**, choose an interesting data set to fit *regression* models for, that is, a data set with a *quantitative* target variable and several feature variables. Could be a data set that: we have used in class, we have used on an assignment, a data set from your project, you used in another course, or you just find interesting! You'll work together with your partner to develop *regression* models to predict the *quantitative* target (so you need to decide on the data set and target variable together with your partner).

After you have decided on the data set and target variable:

- One partner should fit a linear regression model
- One partner should fit a k-nearest neighbors regression model

If you're using an example from earlier in the class, you can use a model you have already come up with or you can try something new. Note: you do NOT have to agree on the features included in your models.

After each partner has their model, **work together to combine your code and create two more models**:

- An ensemble model that uses **voting**
- An ensemble model that uses **stacking**

For each of the four models (linear regression, knn regression, voting ensemble, stacking ensemble)

1. Fit the model using the training data
1. Use the fitted model to predict the target for a few observations of your choice, and compare the predictions
1. Estimate the test RMSE using cross-validation, and compare the results

Which model seems to work best?


In [2]:
#heart attack csv

# %%
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.pipeline import make_pipeline
from sklearn.compose import make_column_transformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LinearRegression
from sklearn.neighbors import KNeighborsRegressor
from sklearn.ensemble import VotingRegressor, StackingRegressor

df = pd.read_csv("heart_attack.csv")
df.head()

# %%
target = "thalach"
features = ["age", "sex", "cp", "trtbps", "chol", "restecg"]
X = df[features]
y = df[target]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# One-hot encode the categorical codes, scale the numeric columns
preprocess = make_column_transformer(
    (OneHotEncoder(handle_unknown="ignore"), ["cp", "restecg"]),
    (StandardScaler(), ["age", "sex", "trtbps", "chol"]),
)

# %% [markdown]
# ### Model 1: Linear regression (Partner A)

# %%
lr_model = make_pipeline(preprocess, LinearRegression())
lr_model.fit(X_train, y_train)

# %% [markdown]
# ### Model 2: k-nearest neighbors regression (Partner B)
# Choose k with cross-validation on the training data.

# %%
knn_pipe = make_pipeline(preprocess, KNeighborsRegressor())
grid = GridSearchCV(
    knn_pipe,
    param_grid={"kneighborsregressor__n_neighbors": range(1, 51)},
    scoring="neg_root_mean_squared_error",
    cv=10,
)
grid.fit(X_train, y_train)
best_k = grid.best_params_["kneighborsregressor__n_neighbors"]
print("Best k:", best_k)

knn_model = make_pipeline(preprocess, KNeighborsRegressor(n_neighbors=best_k))
knn_model.fit(X_train, y_train)

# %% [markdown]
# ### Model 3: Voting ensemble (averages the LR and KNN predictions)

# %%
voting_model = VotingRegressor([
    ("lr", make_pipeline(preprocess, LinearRegression())),
    ("knn", make_pipeline(preprocess, KNeighborsRegressor(n_neighbors=best_k))),
])
voting_model.fit(X_train, y_train)

# %% [markdown]
# ### Model 4: Stacking ensemble (a linear regression learns how to weight LR and KNN)

# %%
stacking_model = StackingRegressor(
    estimators=[
        ("lr", make_pipeline(preprocess, LinearRegression())),
        ("knn", make_pipeline(preprocess, KNeighborsRegressor(n_neighbors=best_k))),
    ],
    final_estimator=LinearRegression(),
    cv=5,
)
stacking_model.fit(X_train, y_train)

# %% [markdown]
# ### Predict a few observations and compare

# %%
models = {
    "Linear Regression": lr_model,
    "KNN": knn_model,
    "Voting": voting_model,
    "Stacking": stacking_model,
}

sample = X_test.head(5)
preds = pd.DataFrame({"actual thalach": y_test.head(5).values}, index=sample.index)
for name, m in models.items():
    preds[name] = m.predict(sample).round(1)
pd.concat([sample, preds], axis=1)

# %% [markdown]
# ### Estimate test RMSE with 10-fold cross-validation

# %%
results = {}
for name, m in models.items():
    scores = cross_val_score(m, X_train, y_train, cv=10,
                             scoring="neg_root_mean_squared_error")
    results[name] = -scores.mean()

cv_rmse = pd.Series(results, name="CV RMSE").sort_values()
cv_rmse




Best k: 44


Stacking             19.212434
Linear Regression    19.212859
Voting               19.243768
KNN                  20.110734
Name: CV RMSE, dtype: float64


# CV RMSE (bpm): Stacking ≈ 19.21, Linear Regression ≈ 19.21, Voting ≈ 19.24, KNN ≈ 20.11.
# Stacking is technically the best, but only by a little bit over plain linear regression.


## Voting in Classification

Suppose you have three models for predicting a categorical target:

- k-nearest neighbors
- logistic regression
- decision tree

How might you apply voting to create an ensemble model? Discuss conceptually with your partner. Create a concrete example to illustrate your method. Hint: There are two basic approaches to voting in classification; can you think of both?

Note: You can assume a binary target, but the general idea of voting applies to multiclass problems. However, if any of the candidate *models* are not naturally multiclass (like logistic regression) then you have to consider things like OvR and OvO as we have discussed.

## Choose your own Classification

**Working together with your partner**, choose an interesting data set to fit *classification* models for, that is, a data set with a *categorical* target variable and several feature variables. Could be a data set that: we have used in class, we have used on an assignment, a data set from your project, you used in another course, or you just find interesting! You'll work together with your partner to develop *classification* models to predict the *categorical* target (so you need to decide on the data set and target variable together with your partner).

After you have decided on the data set and target variable:

- One partner should fit a k-nearest neighbors classification model
- One partner should fit another classification model like decision tree or logistic regression

If you're using an example from earlier in the class, you can use a model you have already come up with or you can try something new. Note: you do NOT have to agree on the features included in your models.

After each partner has their model, **work together to combine your code and create an ensemble model that uses *voting***.

For each of the three models (knn classification, other classification, voting ensemble)

1. Fit the model using the training data
1. Use the fitted model to predict the target for a few observations of your choice, and compare the predictions
1. Estimate the test macro precision, recall, and F1 score using cross-validation, and compare the results

Which model seems to work best?

**Hint:** [`VotingClassifier`](https://scikit-learn.org/1.5/modules/generated/sklearn.ensemble.VotingClassifier.html) works almost identically to `VotingRegressor`.


In [3]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, cross_validate, GridSearchCV
from sklearn.pipeline import make_pipeline
from sklearn.compose import make_column_transformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import VotingClassifier

df = pd.read_csv("NHANES_DXA_body_fat.csv")

# Build the 3-class target and drop the 5 rows with no answer
health_map = {"Excellent": "High", "Very good": "High",
              "Good": "Medium",
              "Fair": "Low", "Poor": "Low"}
df["Health"] = df["SelfRatedHealth"].map(health_map)
df = df.dropna(subset=["Health"])
df["Health"].value_counts()

# %%
num_features = ["AgeYears", "BMI_kg_m2", "WaistCircumference_cm", "DXAPercentBodyFat",
                "UsualWeekdaySleepHours", "AverageSystolicBP_mmHg",
                "TotalCholesterol_mg_dL", "HbA1c_percent", "FamilyIncomePovertyRatio"]
cat_features = ["SexRecorded", "SmokingStatus"]
features = num_features + cat_features

X = df[features]
y = df["Health"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Fill missing values (median for numbers, most common for categories),
# scale numeric columns, and one-hot encode the categorical ones
preprocess = make_column_transformer(
    (make_pipeline(SimpleImputer(strategy="median"), StandardScaler()), num_features),
    (make_pipeline(SimpleImputer(strategy="most_frequent"),
                   OneHotEncoder(handle_unknown="ignore")), cat_features),
)

# %% [markdown]
# ### Model 1: k-nearest neighbors classifier (Partner A)
# Choose k with cross-validation (macro F1) on the training data.

# %%
grid = GridSearchCV(
    make_pipeline(preprocess, KNeighborsClassifier()),
    param_grid={"kneighborsclassifier__n_neighbors": range(1, 51, 2)},
    scoring="f1_macro",
    cv=10,
)
grid.fit(X_train, y_train)
best_k = grid.best_params_["kneighborsclassifier__n_neighbors"]
print("Best k:", best_k)

knn_model = make_pipeline(preprocess, KNeighborsClassifier(n_neighbors=best_k))
knn_model.fit(X_train, y_train)

# %% [markdown]
# ### Model 2: Logistic regression (Partner B)

# %%
logreg_model = make_pipeline(preprocess, LogisticRegression(max_iter=1000))
logreg_model.fit(X_train, y_train)

# %% [markdown]
# ### Model 3: Voting ensemble
# Soft voting averages the two models' predicted probabilities for each class.

# %%
voting_model = VotingClassifier(
    estimators=[
        ("knn", make_pipeline(preprocess, KNeighborsClassifier(n_neighbors=best_k))),
        ("logreg", make_pipeline(preprocess, LogisticRegression(max_iter=1000))),
    ],
    voting="soft",
)
voting_model.fit(X_train, y_train)

# %% [markdown]
# ### Predict a few observations and compare

# %%
models = {
    "KNN": knn_model,
    "Logistic Regression": logreg_model,
    "Voting": voting_model,
}

sample = X_test.head(8)
preds = pd.DataFrame({"actual": y_test.head(8).values}, index=sample.index)
for name, m in models.items():
    preds[name] = m.predict(sample)
pd.concat([sample[["AgeYears", "SexRecorded", "BMI_kg_m2", "HbA1c_percent",
                   "SmokingStatus"]], preds], axis=1)

# %% [markdown]
# ### Estimate test macro precision, recall, and F1 with 10-fold cross-validation

# %%
results = {}
for name, m in models.items():
    cv = cross_validate(m, X_train, y_train, cv=10,
                        scoring=["precision_macro", "recall_macro", "f1_macro"])
    results[name] = {
        "Precision (macro)": cv["test_precision_macro"].mean(),
        "Recall (macro)": cv["test_recall_macro"].mean(),
        "F1 (macro)": cv["test_f1_macro"].mean(),
    }

cv_results = pd.DataFrame(results).T.sort_values("F1 (macro)", ascending=False).round(3)
cv_results

Best k: 41


,Precision (macro),Recall (macro),F1 (macro)
Logistic Regression,0.519,0.478,0.481
Voting,0.536,0.479,0.476
KNN,0.522,0.469,0.463


logistic regression works best because it has the highest macro F1 and recall